# 23.1 Normal-form games & Nash equilibrium

A Nash equilibrium is not a globally best outcome; it is a resting point where no player can gain by moving alone. This lesson uses a payoff table where probability becomes a mixed strategy, optimization becomes best response, and equilibrium is a fixed point of mutual best responses. Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 231
random.seed(SEED)
np.random.seed(SEED)
TOL = 1e-8

## The concept, built once: best responses and mixed indifference

For a two-player game with payoff matrices $A$ and $B$, a pure profile $(i,j)$ is a Nash equilibrium when row $i$ maximizes $A_{ij}$ in column $j$ and column $j$ maximizes $B_{ij}$ in row $i$.

The lesson prisoner's-dilemma numbers are $A=\begin{bmatrix}3&0\\5&1\end{bmatrix}$ and $B=\begin{bmatrix}3&5\\0&1\end{bmatrix}$, so only $(D,D)$ survives with payoffs $(1,1)$.

In [ ]:
def best_response_sets(A, B, tol=1e-8):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    row_sets = []
    col_sets = []

    for j in range(A.shape[1]):
        column = A[:, j]
        best = float(np.max(column))
        rows = [i for i, value in enumerate(column) if value >= best - tol]
        row_sets.append(rows)

    for i in range(B.shape[0]):
        row = B[i, :]
        best = float(np.max(row))
        cols = [j for j, value in enumerate(row) if value >= best - tol]
        col_sets.append(cols)

    return row_sets, col_sets


def find_pure_nash(A, B, tol=1e-8):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    row_sets, col_sets = best_response_sets(A, B, tol=tol)
    equilibria = []

    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            row_ok = i in row_sets[j]
            col_ok = j in col_sets[i]
            if row_ok and col_ok:
                equilibria.append((i, j, float(A[i, j]), float(B[i, j])))

    return equilibria


pd_A = np.array([[3, 0], [5, 1]], dtype=float)
pd_B = np.array([[3, 5], [0, 1]], dtype=float)
pd_equilibria = find_pure_nash(pd_A, pd_B)

print(pd_equilibria)
assert pd_equilibria == [(1, 1, 1.0, 1.0)]
assert pd_A[0, 0] + pd_B[0, 0] == 6
assert pd_A[1, 1] + pd_B[1, 1] == 2

Now add the mixed-strategy condition. In a $2\times2$ row payoff matrix $A$, if the column player uses probability $q$ on Left, the row player is indifferent when

$$A_{00}q + A_{01}(1-q) = A_{10}q + A_{11}(1-q).$$

For the lesson matrix $\begin{bmatrix}2&0\\0&1\end{bmatrix}$, this gives $2q=1-q$, so $q=0.333333$ and each supported row action earns $0.666667$.

In [ ]:
def mixed_indifference_2x2(A):
    A = np.asarray(A, dtype=float)
    numerator = A[1, 1] - A[0, 1]
    denominator = A[0, 0] - A[0, 1] - A[1, 0] + A[1, 1]
    q = numerator / denominator
    top_value = A[0, 0] * q + A[0, 1] * (1 - q)
    bottom_value = A[1, 0] * q + A[1, 1] * (1 - q)
    return float(q), float(top_value), float(bottom_value)


def solve_support_probability(payoff_matrix, supported_rows, supported_cols):
    payoff_matrix = np.asarray(payoff_matrix, dtype=float)
    anchor = supported_rows[0]
    rows = []
    rhs = []

    for row_index in supported_rows[1:]:
        row = payoff_matrix[row_index, supported_cols] - payoff_matrix[anchor, supported_cols]
        rows.append(row)
        rhs.append(0.0)

    rows.append(np.ones(len(supported_cols)))
    rhs.append(1.0)
    solution = np.linalg.solve(np.vstack(rows), np.array(rhs))
    return solution


def support_enumeration_nash(A, B, tol=1e-7):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    equilibria = []
    row_count = A.shape[0]
    col_count = A.shape[1]
    max_support = min(row_count, col_count)

    for support_size in range(1, max_support + 1):
        row_supports = itertools.combinations(range(row_count), support_size)
        col_supports = itertools.combinations(range(col_count), support_size)
        for row_support in row_supports:
            for col_support in itertools.combinations(range(col_count), support_size):
                row_support = list(row_support)
                col_support = list(col_support)
                try:
                    q_support = solve_support_probability(A, row_support, col_support)
                    p_support = solve_support_probability(B.T, col_support, row_support)
                except np.linalg.LinAlgError:
                    continue
                if np.min(q_support) < -tol or np.min(p_support) < -tol:
                    continue
                p = np.zeros(row_count)
                q = np.zeros(col_count)
                p[row_support] = np.maximum(p_support, 0.0)
                q[col_support] = np.maximum(q_support, 0.0)
                p = p / np.sum(p)
                q = q / np.sum(q)
                row_values = A @ q
                col_values = p @ B
                row_value = float(row_values[row_support[0]])
                col_value = float(col_values[col_support[0]])
                row_ok = np.max(row_values) <= row_value + tol
                col_ok = np.max(col_values) <= col_value + tol
                if row_ok and col_ok:
                    equilibria.append((p, q, row_value, col_value))

    return equilibria


lesson_mixed = np.array([[2, 0], [0, 1]], dtype=float)
q, top_value, bottom_value = mixed_indifference_2x2(lesson_mixed)
print(q, top_value, bottom_value)
assert round(q, 6) == 0.333333
assert round(top_value, 6) == 0.666667
assert round(bottom_value, 6) == 0.666667

## The dataset ladder: D1 to D5 game/profile instances

The ladder is inline for family F16: rungs grow from a hand-solvable $2\times2$ game to a harder mixed/tie-heavy $4\times4$ game.

In [ ]:
def build_normal_form_ladder():
    ladder = []
    ladder.append({
        "name": "D1 prisoner's dilemma",
        "A": np.array([[3, 0], [5, 1]], dtype=float),
        "B": np.array([[3, 5], [0, 1]], dtype=float),
        "labels": ["C", "D"],
    })
    ladder.append({
        "name": "D2 coordination and anti-coordination tensions",
        "A": np.array([[4, 0], [1, 3]], dtype=float),
        "B": np.array([[4, 1], [0, 3]], dtype=float),
        "labels": ["Invest", "Wait"],
    })
    ladder.append({
        "name": "D3 matching pennies cycle",
        "A": np.array([[1, -1], [-1, 1]], dtype=float),
        "B": np.array([[-1, 1], [1, -1]], dtype=float),
        "labels": ["Heads", "Tails"],
    })
    ladder.append({
        "name": "D4 tie-heavy 3x3 bimatrix",
        "A": np.array([[3, 3, 0], [1, 2, 2], [3, 1, 2]], dtype=float),
        "B": np.array([[2, 2, 1], [3, 1, 3], [0, 3, 2]], dtype=float),
        "labels": ["A0", "A1", "A2"],
    })
    ladder.append({
        "name": "D5 cyclic 4x4 mixed-equilibrium game",
        "A": np.array([[0, -1, 1, 0], [1, 0, -1, 0], [-1, 1, 0, 0], [0, 0, 0, 0]], dtype=float),
        "B": np.array([[0, 1, -1, 0], [-1, 0, 1, 0], [1, -1, 0, 0], [0, 0, 0, 0]], dtype=float),
        "labels": ["Rock", "Paper", "Scissors", "Safe"],
    })
    return ladder


normal_ladder = build_normal_form_ladder()
for rung in normal_ladder:
    print(rung["name"], rung["A"].shape, "sample", rung["A"][0].tolist())

## Run the SAME method across D1-D5

Use the same equilibrium solver on every rung and collect max unilateral regret plus equilibrium counts.

In [ ]:
def max_unilateral_regret(A, B, p, q):
    row_values = A @ q
    col_values = p @ B
    row_payoff = float(p @ A @ q)
    col_payoff = float(p @ B @ q)
    row_regret = float(np.max(row_values) - row_payoff)
    col_regret = float(np.max(col_values) - col_payoff)
    return max(row_regret, col_regret)


results = []
for rung in normal_ladder:
    A = rung["A"]
    B = rung["B"]
    pure = find_pure_nash(A, B)
    mixed = support_enumeration_nash(A, B)
    if mixed:
        p, q, row_value, col_value = mixed[0]
        regret = max_unilateral_regret(A, B, p, q)
    else:
        p = np.ones(A.shape[0]) / A.shape[0]
        q = np.ones(A.shape[1]) / A.shape[1]
        regret = max_unilateral_regret(A, B, p, q)
    results.append({
        "name": rung["name"],
        "size": A.shape[0] * A.shape[1],
        "pure_count": len(pure),
        "mixed_count": len(mixed),
        "regret": regret,
    })

for row in results:
    print(row["name"], "pure", row["pure_count"], "mixed", row["mixed_count"], "regret", round(row["regret"], 6))

## Results visualization

The closing figure has payoff-matrix panels plus a metric curve over rung size.

In [ ]:
fig, axes = plt.subplots(2, len(normal_ladder), figsize=(18, 7))

for axis, rung in zip(axes[0], normal_ladder):
    A = rung["A"]
    B = rung["B"]
    social = A + B
    image = axis.imshow(social, cmap="viridis")
    pure = find_pure_nash(A, B)
    for i, j, row_payoff, col_payoff in pure:
        axis.scatter(j, i, s=180, facecolors="none", edgecolors="white", linewidths=2)
    axis.set_title(rung["name"].split()[0] + " social payoff")
    axis.set_xlabel("Column action")
    axis.set_ylabel("Row action")

sizes = [row["size"] for row in results]
regrets = [row["regret"] for row in results]
counts = [row["pure_count"] + row["mixed_count"] for row in results]
axes[1, 0].plot(sizes, regrets, marker="o", label="max unilateral regret")
axes[1, 0].set_xlabel("cells in payoff matrix")
axes[1, 0].set_ylabel("regret")
axes[1, 0].legend()
axes[1, 1].plot(sizes, counts, marker="s", color="tab:orange", label="equilibria found")
axes[1, 1].set_xlabel("cells in payoff matrix")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()
for axis in axes[1, 2:]:
    axis.axis("off")
fig.colorbar(image, ax=axes[0].ravel().tolist(), shrink=0.6)
plt.tight_layout()
plt.show()

## Pitfall on D5: maximizing total payoff is not equilibrium

The lesson warns against calling the best total-payoff cell a Nash equilibrium. On D5, reproduce that wrong rule, then fix it by checking unilateral deviations and tie-aware supports.

In [ ]:
d5 = normal_ladder[-1]
A = d5["A"]
B = d5["B"]
social = A + B
wrong_cell = np.unravel_index(np.argmax(social), social.shape)
wrong_p = np.eye(A.shape[0])[wrong_cell[0]]
wrong_q = np.eye(A.shape[1])[wrong_cell[1]]
wrong_regret = max_unilateral_regret(A, B, wrong_p, wrong_q)
fixed_equilibria = support_enumeration_nash(A, B)
fixed_p, fixed_q, row_value, col_value = fixed_equilibria[0]
fixed_regret = max_unilateral_regret(A, B, fixed_p, fixed_q)

print("wrong social-optimum cell", wrong_cell, "regret", round(wrong_regret, 6))
print("fixed mixed p", np.round(fixed_p, 3), "q", np.round(fixed_q, 3), "regret", round(fixed_regret, 6))
assert fixed_regret <= wrong_regret + TOL

## Evaluate it + Practice

- Metric: max unilateral regret should be near zero for an equilibrium; compare with a uniform random no-skill baseline.
- Sanity check: every pure equilibrium returned by `find_pure_nash` must be a mutual best-response cell.
- Ablation: drop tie-aware best responses and D4 can lose valid equilibria.
- Failure signal: best-response dynamics can cycle on D3 even when the support-enumeration solver finds a mixed equilibrium.

Practice 1: Change the D2 payoff table so one coordinated outcome is risk-dominant and recompute pure equilibria.

Practice 2: Add a dominated row action to D5 and verify the mixed support drops it.